# Model Loss Check (joblib)

This notebook loads the saved models from the `models/` folder and reports loss/metrics on the current test split.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    log_loss,
    accuracy_score,
    precision_score,
    f1_score,
)

from src.data_loader import DataLoader
from src.models.meta_filter import MetaFilter

In [2]:
root = Path('.').resolve()
models_dir = root / 'models'
xgb_path = models_dir / 'xgboost_AAPL.joblib'
meta_path = models_dir / 'meta_rf_AAPL.joblib'

def unwrap_model(obj):
    if isinstance(obj, dict) and 'model' in obj:
        return obj['model'], obj
    return obj, {}

xgb_data = joblib.load(xgb_path)
meta_data = joblib.load(meta_path)

xgb_model, xgb_info = unwrap_model(xgb_data)
meta_model, meta_info = unwrap_model(meta_data)

xgb_features = xgb_info.get('feature_names')
meta_features = meta_info.get('feature_names')

print('Loaded models:')
print(' - XGB:', type(xgb_model))
print(' - Meta:', type(meta_model))
print('XGB features:', xgb_features)
print('Meta features:', meta_features)

Loaded models:
 - XGB: <class 'xgboost.sklearn.XGBRegressor'>
 - Meta: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
XGB features: ['VXN', 'DXY', 'BB_PB', 'EMA50', 'Close']
Meta features: ['VXN', 'DXY', 'BB_PB', 'EMA50', 'Close', 'Predicted_Price', 'Implied_Signal', 'Model_Confidence']


In [3]:
# Load data (same split as pipeline)
loader = DataLoader(ticker='AAPL')
data = loader.load()

df_test = data.df_test.copy()

# Use model's stored feature names if available
if xgb_features is None:
    xgb_features = list(data.feature_columns)

X_test = df_test[xgb_features]
y_test = df_test['Target']

xgb_pred = xgb_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
mae = mean_absolute_error(y_test, xgb_pred)
r2 = r2_score(y_test, xgb_pred)

print('XGBoost loss/metrics on test:')
print(f'  RMSE: {rmse:.6f}')
print(f'  MAE:  {mae:.6f}')
print(f'  R²:   {r2:.4f}')

Loading AAPL...
  Fetched AAPL: 6559 rows
  Fetched VXN: 6293 rows
  Fetched DXY: 6588 rows
  Train: 4513 rows
  Val:   757 rows
  Test:  1022 rows
XGBoost loss/metrics on test:
  RMSE: 196.625212
  MAE:  192.995606
  R²:   -26.3386


In [4]:
# Meta-Filter evaluation (classification loss/metrics)
meta = MetaFilter()
meta.model = meta_model
meta.is_fitted = True
meta.feature_names = meta_features or MetaFilter.META_FEATURES

X_meta_test = meta.prepare_features(df_test, xgb_pred)
y_meta_test = meta.create_target(df_test, xgb_pred)

meta_pred = meta.predict(X_meta_test)

if hasattr(meta_model, 'predict_proba'):
    proba = meta_model.predict_proba(X_meta_test)[:, 1]
    ll = log_loss(y_meta_test, proba, labels=[0, 1])
else:
    ll = float('nan')

acc = accuracy_score(y_meta_test, meta_pred)
prec = precision_score(y_meta_test, meta_pred, zero_division=0)
f1 = f1_score(y_meta_test, meta_pred, zero_division=0)

print('Meta-Filter loss/metrics on test:')
print(f'  Log Loss: {ll:.6f}')
print(f'  Accuracy: {acc:.2%}')
print(f'  Precision: {prec:.2%}')
print(f'  F1: {f1:.2%}')

Meta-Filter loss/metrics on test:
  Log Loss: 0.674804
  Accuracy: 56.65%
  Precision: 53.05%
  F1: 63.36%
